# GLTC-TNN 黑白视频修复算法
几何低秩张量补全 - 截断核范数 (Geometric Low-rank Tensor Completion - Truncated Nuclear Norm)


In [ ]:
import numpy as np
from skimage.metrics import structural_similarity as ssim
import matplotlib.pyplot as plt
import os
import time
import pandas as pd

# 绘图字体设置
# 中文优先使用宋体，英文和数字使用 Times New Roman
plt.rcParams['font.sans-serif'] = ['SimSun', 'Times New Roman']
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.unicode_minus'] = False

def ten2mat(tensor, mode):
    """张量按模展开为矩阵"""
    return np.reshape(np.moveaxis(tensor, mode, 0), (tensor.shape[mode], -1), order='F')

def mat2ten(mat, tensor_size, mode):
    """矩阵折叠为张量"""
    index = list()
    index.append(mode)
    for i in range(tensor_size.shape[0]):
        if i != mode:
            index.append(i)
    return np.moveaxis(np.reshape(mat, list(tensor_size[index]), order='F'), 0, mode)

def svt_tnn(mat, tau, theta):
    """截断核范数的奇异值阈值化"""
    u, s, v = np.linalg.svd(mat, full_matrices=False)
    vec = np.zeros(len(s))
    for i in range(len(s)):
        if i >= theta:
            vec[i] = s[i] - tau
        else:
            vec[i] = s[i]
    vec[vec < 0] = 0
    return np.matmul(np.matmul(u, np.diag(vec)), v)

def read_yuv420_gray(video_path, height, width):
    """读取 QCIF 黑白视频。

    优先按照 YUV420 文件读取，只使用 Y 分量。
    如果文件是纯灰度 raw yuv，也会自动兼容。
    """
    y_size = width * height
    uv_size = y_size // 4
    yuv420_frame_size = y_size + 2 * uv_size
    file_size = os.path.getsize(video_path)

    if file_size % yuv420_frame_size == 0:
        frame_num = file_size // yuv420_frame_size
        video = np.zeros((height, width, frame_num), dtype=np.uint8)
        with open(video_path, 'rb') as f:
            for k in range(frame_num):
                y = np.frombuffer(f.read(y_size), dtype=np.uint8)
                if y.size != y_size:
                    raise ValueError(f'读取第 {k + 1} 帧失败: {video_path}')
                video[:, :, k] = y.reshape((height, width))
                f.seek(2 * uv_size, 1)
        return video, frame_num

    if file_size % y_size == 0:
        frame_num = file_size // y_size
        video = np.fromfile(video_path, dtype=np.uint8)
        video = video.reshape((frame_num, height, width))
        video = np.transpose(video, (1, 2, 0))
        return video, frame_num

    raise ValueError(f'视频大小与 QCIF 灰度或 YUV420 格式不匹配: {video_path}')

def write_yuv420_gray(video, save_path):
    """把修复后的灰度视频保存为 YUV420 文件，UV 分量写为 128。"""
    video = np.clip(np.round(video), 0, 255).astype(np.uint8)
    height, width, frame_num = video.shape
    uv_size = height * width // 4

    with open(save_path, 'wb') as f:
        for k in range(frame_num):
            f.write(video[:, :, k].tobytes())
            f.write(np.full(uv_size, 128, dtype=np.uint8).tobytes())
            f.write(np.full(uv_size, 128, dtype=np.uint8).tobytes())

def calculate_video_psnr(original, recovered, maxP=255.0):
    """按帧计算 PSNR 后取平均。"""
    original = original.astype(float)
    recovered = recovered.astype(float)
    mse_per_frame = np.mean((original - recovered) ** 2, axis=(0, 1))
    psnr_per_frame = np.full_like(mse_per_frame, np.inf, dtype=float)
    valid = mse_per_frame > 0
    psnr_per_frame[valid] = 10 * np.log10(maxP ** 2 / mse_per_frame[valid])

    finite_values = psnr_per_frame[np.isfinite(psnr_per_frame)]
    if finite_values.size == 0:
        return np.inf
    return float(np.mean(finite_values))

def calculate_video_ssim(original, recovered, maxP=255.0):
    """逐帧计算 SSIM 后取平均。"""
    original = np.clip(original, 0, 255).astype(float)
    recovered = np.clip(recovered, 0, 255).astype(float)
    frame_num = original.shape[2]
    values = []

    for k in range(frame_num):
        value = ssim(original[:, :, k], recovered[:, :, k], data_range=maxP)
        if np.isfinite(value):
            values.append(value)

    if len(values) == 0:
        return np.nan
    return float(np.mean(values))

def safe_float(value):
    try:
        return float(value)
    except Exception:
        return np.nan

def safe_save_summary(summary_rows, result_dir, filename='实验总结.xlsx'):
    """优先保存 Excel；如果 Excel 写入失败，则自动保存 CSV。"""
    os.makedirs(result_dir, exist_ok=True)
    summary_df = pd.DataFrame(summary_rows)
    summary_df = summary_df.replace([np.inf, -np.inf], np.nan)

    excel_path = os.path.join(result_dir, filename)
    csv_path = os.path.join(result_dir, filename.replace('.xlsx', '.csv'))

    try:
        summary_df.to_excel(excel_path, index=False, engine='openpyxl', na_rep='NaN', inf_rep='Inf')
        return excel_path
    except Exception as e:
        summary_df.to_csv(csv_path, index=False, encoding='utf-8-sig', na_rep='NaN')
        error_path = os.path.join(result_dir, 'excel写入失败说明.txt')
        with open(error_path, 'w', encoding='utf-8') as f:
            f.write('Excel 写入失败，已改为保存 CSV。\n')
            f.write(f'错误信息: {repr(e)}\n')
        print(f'Excel 写入失败，已保存 CSV: {csv_path}')
        return csv_path


In [ ]:
def GLTC_TNN(dense_tensor, sparse_tensor, Omega, alpha, beta, rho, theta, maxiter, verbose=True):
    """
    GLTC-TNN 算法主函数

    参数:
    dense_tensor: 原始黑白视频张量，尺寸为 height × width × frames
    sparse_tensor: 观测到的稀疏视频张量
    Omega: 二值掩码，1 表示观测，0 表示缺失
    alpha: 正则化参数
    beta: 图正则化参数
    rho: ADMM 惩罚参数
    theta: 截断参数，保留前 theta 个奇异值
    maxiter: 最大迭代次数
    verbose: 是否打印进度

    返回:
    tensor_hat: 修复后的视频张量
    psnr_list: 每次迭代的 PSNR
    ssim_value: 最终平均 SSIM
    elapsed_time: 运行时间
    """
    Omega = Omega.astype(bool)
    dim0 = sparse_tensor.ndim
    dim1, dim2, dim3 = sparse_tensor.shape
    dim = np.array([dim1, dim2, dim3])
    maxP = 255.0

    binary_tensor = Omega.astype(float)
    sparse_tensor = sparse_tensor.astype(float)
    tensor_hat = sparse_tensor.copy()

    X = np.zeros((dim1, dim2, dim3, dim0))
    Z = np.zeros((dim1, dim2, dim3, dim0))
    T = np.zeros((dim1, dim2, dim3, dim0))
    for k in range(dim0):
        X[:, :, :, k] = tensor_hat
        Z[:, :, :, k] = tensor_hat

    # 构造空间方向的邻接平滑矩阵
    D1 = np.zeros((dim1 - 1, dim1))
    for i in range(dim1 - 1):
        D1[i, i] = -1
        D1[i, i + 1] = 1

    D2 = np.zeros((dim2 - 1, dim2))
    for i in range(dim2 - 1):
        D2[i, i] = -1
        D2[i, i + 1] = 1

    psnr_list = []
    start_time = time.time()

    for iters in range(maxiter):
        # 更新 Z 和 X
        for k in range(dim0):
            Z_hat = ten2mat(X[:, :, :, k] + T[:, :, :, k] / rho, k)
            Z_hat = svt_tnn(Z_hat, alpha / rho, theta)
            Z[:, :, :, k] = mat2ten(Z_hat, dim, k)

            var = ten2mat(rho * Z[:, :, :, k] - T[:, :, :, k], k)
            if k == 0:
                var0 = mat2ten(
                    np.matmul(np.linalg.inv(beta * np.matmul(D1.T, D1) + rho * np.eye(dim1)), var),
                    dim,
                    k
                )
            elif k == 1:
                var0 = mat2ten(
                    np.matmul(np.linalg.inv(beta * np.matmul(D2.T, D2) + rho * np.eye(dim2)), var),
                    dim,
                    k
                )
            else:
                var0 = Z[:, :, :, k] - T[:, :, :, k] / rho

            X[:, :, :, k] = np.multiply(1 - binary_tensor, var0) + sparse_tensor

        tensor_hat = np.mean(X, axis=3)
        tensor_hat = np.clip(tensor_hat, 0, maxP)

        current_psnr = calculate_video_psnr(dense_tensor, tensor_hat, maxP=maxP)
        psnr_list.append(current_psnr)

        if verbose:
            print(f"Epoch = {iters + 1}, PSNR = {current_psnr:.4f} dB")

        # 更新拉格朗日乘子和 X
        for k in range(dim0):
            T[:, :, :, k] = T[:, :, :, k] + rho * (X[:, :, :, k] - Z[:, :, :, k])
            X[:, :, :, k] = tensor_hat.copy()

    psnr_list = np.asarray(psnr_list, dtype=float)
    tensor_hat = np.nan_to_num(tensor_hat, nan=0.0, posinf=maxP, neginf=0.0)
    tensor_hat = np.clip(tensor_hat, 0, maxP)
    ssim_value = calculate_video_ssim(dense_tensor, tensor_hat, maxP=maxP)
    elapsed_time = time.time() - start_time

    if verbose:
        print(f"\n{'=' * 50}")
        print(f"最终 PSNR: {psnr_list[-1]:.4f} dB")
        print(f"最终 SSIM: {ssim_value:.4f}")
        print(f"运行时间: {elapsed_time:.2f} 秒")
        print(f"{'=' * 50}")

    return tensor_hat, psnr_list, ssim_value, elapsed_time


In [ ]:
# 参数设置
seedr = 920
np.random.seed(seedr)

# 需要依次运行的视频文件
video_files = [
    {
        'video_name': 'news_qcif',
        'filename': 'news_qcif_gray.yuv'
    },
    {
        'video_name': 'akiyo_qcif',
        'filename': 'akiyo_qcif_gray.yuv'
    }
]

# 自动识别当前运行目录
base_dir = os.getcwd()
candidate_dirs = [
    base_dir,
    os.path.join(base_dir, '视频实验')
]

video_dir = None
for candidate_dir in candidate_dirs:
    processed_dir = os.path.join(candidate_dir, '处理后视频')
    if all(os.path.exists(os.path.join(processed_dir, item['filename'])) for item in video_files):
        video_dir = candidate_dir
        break

if video_dir is None:
    raise FileNotFoundError('没有找到同时包含 news_qcif_gray.yuv 和 akiyo_qcif_gray.yuv 的处理后视频目录')

# 视频格式设置
w = 176
h = 144
fps = 30
show_idx = [40, 200]

# 算法参数，保持 GLTC-TNN 当前参数不变
missing_rates = [0.8, 0.9, 0.95]
theta = 100
alpha = 10
rho = 1.001
beta = 0.1 * rho
maxiter = 1000

# 存储结果
all_results = []

for video_item in video_files:
    video_name = video_item['video_name']
    video_path = os.path.join(video_dir, '处理后视频', video_item['filename'])
    I, frame_num = read_yuv420_gray(video_path, h, w)

    print(f"\n{'#' * 70}")
    print(f"开始处理视频: {video_item['filename']}")
    print(f"视频路径: {video_path}")
    print(f"尺寸: {h} x {w} x {frame_num}")
    print(f"{'#' * 70}\n")

    for missing_rate in missing_rates:
        np.random.seed(seedr)
        Omega = np.random.rand(h, w, frame_num) > missing_rate
        observed = (I.astype(float) * Omega).astype(np.uint8)

        result_dir = os.path.join(
            video_dir,
            'results',
            'GLTC-TNN',
            f'{video_name}_{int(missing_rate * 100)}'
        )
        os.makedirs(result_dir, exist_ok=True)

        print(f"\n{'=' * 60}")
        print('GLTC-TNN 黑白视频修复实验')
        print(f"{'=' * 60}")
        print(f"视频: {video_path}")
        print(f"尺寸: {h} x {w} x {frame_num}")
        print(f"帧率: {fps}")
        print(f"缺失率: {missing_rate * 100:.1f}%")
        print(f"观测像素: {np.sum(Omega)} / {I.size}")
        print(f"输出目录: {result_dir}")
        print(f"参数: alpha={alpha}, beta={beta}, rho={rho}, theta={theta}, maxiter={maxiter}")
        print(f"{'=' * 60}\n")

        status = 'ok'
        error_msg = ''

        try:
            X_gltctnn, psnr_gltctnn, ssim_gltctnn, elapsed_time = GLTC_TNN(
                I, observed, Omega, alpha, beta, rho, theta, maxiter
            )

            X_gltctnn = np.nan_to_num(X_gltctnn, nan=0.0, posinf=255.0, neginf=0.0)
            X_gltctnn = np.clip(np.round(X_gltctnn), 0, 255).astype(np.uint8)
            final_psnr = calculate_video_psnr(I, X_gltctnn, maxP=255.0)
            final_ssim = calculate_video_ssim(I, X_gltctnn, maxP=255.0)

            if not np.isfinite(final_psnr):
                status = 'nan_metric'

        except Exception as e:
            status = 'error'
            error_msg = repr(e)
            print(f"GLTC-TNN 运行失败，将保存观测视频作为占位结果，并继续后续实验。错误: {error_msg}")

            X_gltctnn = observed.copy()
            psnr_gltctnn = np.array([], dtype=float)
            final_psnr = np.nan
            final_ssim = np.nan
            elapsed_time = 0.0

        best_result = {
            'original': I,
            'observed': observed,
            'recovered': X_gltctnn,
            'psnr_list': psnr_gltctnn,
            'final_psnr': final_psnr,
            'ssim': final_ssim,
            'time': elapsed_time,
            'status': status
        }

        summary_rows = [
            {
                'Video': video_name,
                'MissingRate': missing_rate,
                'alpha': alpha,
                'beta': beta,
                'rho': rho,
                'theta': theta,
                'maxiter': maxiter,
                'PSNR': final_psnr,
                'SSIM': final_ssim,
                'Time': elapsed_time,
                'Status': status,
                'Error': error_msg
            }
        ]

        print(f"最终 PSNR: {final_psnr:.4f} dB" if np.isfinite(final_psnr) else "最终 PSNR: NaN")
        print(f"最终 SSIM: {final_ssim:.4f}" if np.isfinite(final_ssim) else "最终 SSIM: NaN")
        print(f"运行时间: {elapsed_time:.2f} 秒")
        print(f"状态: {status}")

        # 保存修复视频和原始结果
        write_yuv420_gray(best_result['recovered'], os.path.join(result_dir, f'{video_name}_recovered.yuv'))
        np.savez(
            os.path.join(result_dir, 'result.npz'),
            video_name=video_name,
            video_path=video_path,
            I=I,
            Omega=Omega,
            observed=observed,
            recovered=best_result['recovered'],
            psnr_list=best_result['psnr_list'],
            alpha=alpha,
            beta=beta,
            rho=rho,
            theta=theta,
            maxiter=maxiter
        )

        # 保存指标
        metrics_path = os.path.join(result_dir, 'metrics.txt')
        with open(metrics_path, 'w', encoding='utf-8') as f:
            f.write(f'video_name={video_name}\n')
            f.write(f'video={video_path}\n')
            f.write(f'size={h} x {w} x {frame_num}\n')
            f.write(f'fps={fps}\n')
            f.write(f'missing_rate={missing_rate:.2f}\n')
            f.write(f'alpha={alpha}\n')
            f.write(f'beta={beta}\n')
            f.write(f'rho={rho}\n')
            f.write(f'theta={theta}\n')
            f.write(f'maxiter={maxiter}\n')
            f.write(f'psnr={safe_float(best_result["final_psnr"]):.6f}\n' if np.isfinite(safe_float(best_result['final_psnr'])) else 'psnr=NaN\n')
            f.write(f'ssim={safe_float(best_result["ssim"]):.6f}\n' if np.isfinite(safe_float(best_result['ssim'])) else 'ssim=NaN\n')
            f.write(f'time={safe_float(best_result["time"]):.6f}\n' if np.isfinite(safe_float(best_result['time'])) else 'time=NaN\n')
            f.write(f'status={best_result["status"]}\n')
            f.write(f'error={error_msg}\n')

        summary_path = safe_save_summary(summary_rows, result_dir)
        print(f"指标表已保存: {summary_path}")

        # 保存指定帧对比图
        show_idx_valid = [idx for idx in show_idx if idx <= frame_num]
        for idx in show_idx_valid:
            ori = I[:, :, idx - 1]
            obs = observed[:, :, idx - 1]
            rec = best_result['recovered'][:, :, idx - 1]

            fig, axes = plt.subplots(1, 3, figsize=(12, 4))

            axes[0].imshow(ori, cmap='gray')
            axes[0].set_title(f'原始视频帧 {idx}', fontsize=14)
            axes[0].axis('off')

            axes[1].imshow(obs, cmap='gray')
            axes[1].set_title(f'观测数据 ({missing_rate * 100:.0f}% 缺失)', fontsize=14)
            axes[1].axis('off')

            axes[2].imshow(rec, cmap='gray')
            axes[2].set_title(
                f'GLTC-TNN 修复\nPSNR={best_result["final_psnr"]:.2f}dB, SSIM={best_result["ssim"]:.4f}\n时间={best_result["time"]:.2f}s',
                fontsize=14
            )
            axes[2].axis('off')

            plt.tight_layout()
            plt.savefig(os.path.join(result_dir, f'frame_{idx:03d}_compare.png'), dpi=150, bbox_inches='tight')
            plt.show()

        # 保存收敛曲线
        plt.figure(figsize=(10, 6))
        psnr_list = np.asarray(best_result['psnr_list'], dtype=float)
        if psnr_list.size > 0:
            plt.plot(range(1, len(psnr_list) + 1), psnr_list, linewidth=2)
        else:
            plt.text(0.5, 0.5, '没有可用的 PSNR 曲线', ha='center', va='center', transform=plt.gca().transAxes)

        plt.xlabel('迭代次数', fontsize=12)
        plt.ylabel('PSNR (dB)', fontsize=12)
        plt.title(
            f'{video_name} - GLTC-TNN 收敛曲线\n'
            f'missing_rate={missing_rate:.2f}, alpha={alpha}, beta={beta}, rho={rho}',
            fontsize=14
        )
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(result_dir, '收敛曲线.png'), dpi=150, bbox_inches='tight')
        plt.show()

        print(f"\n视频 {video_name}，缺失率 {missing_rate * 100:.1f}% 的结果已保存")

        all_results.append({
            'video_name': video_name,
            'video_path': video_path,
            'missing_rate': missing_rate,
            'result_dir': result_dir,
            'alpha': alpha,
            'beta': beta,
            'rho': rho,
            'theta': theta,
            'maxiter': maxiter,
            'best_result': best_result,
            'summary_rows': summary_rows
        })

print(f"\n{'=' * 60}")
print('所有视频和缺失率处理完成！')
print(f"{'=' * 60}")


In [ ]:
# 单独导出总 Excel 汇总
if 'all_results' not in globals():
    raise NameError('当前内核中没有 all_results，请先运行上一单元生成实验结果。')

total_summary = []

for item in all_results:
    data = item['best_result']
    total_summary.append({
        'Video': item['video_name'],
        'MissingRate': item['missing_rate'],
        'alpha': item['alpha'],
        'beta': item['beta'],
        'rho': item['rho'],
        'theta': item['theta'],
        'maxiter': item['maxiter'],
        'PSNR': data['final_psnr'],
        'SSIM': data['ssim'],
        'Time': data['time'],
        'Status': data['status'],
        'ResultDir': item['result_dir']
    })

total_result_dir = os.path.join(video_dir, 'results', 'GLTC-TNN')
summary_path = safe_save_summary(total_summary, total_result_dir, filename='全部实验总结.xlsx')
print(f'总 Excel 已导出到: {summary_path}')
